### 0. Setup

In [ ]:
# All imports used by this notebook.
from pathlib import Path
import csv
import importlib.util
import inspect
import json
import os
import shutil
import subprocess
import sys
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pprint import pprint
from typing import Any

import yaml


def import_module_from_path(module_name: str, path: Path):
    """Import a Python module from an explicit file path, always reloading from disk."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Module file not found: {path}")

    # Avoid accidentally reusing an older imported converter from sys.modules.
    sys.modules.pop(module_name, None)

    spec = importlib.util.spec_from_file_location(module_name, str(path))
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not create import spec for {path}")

    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


def add_to_sys_path(path: Path) -> None:
    """Prepend a path to sys.path if it is not already present."""
    path_str = str(Path(path).resolve())
    if path_str not in sys.path:
        sys.path.insert(0, path_str)


def ensure_fsp_by_resp_id(path: Path) -> bool:
    """
    Ensure a ChainScope CotResponses YAML file contains fsp_by_resp_id.

    For these Qwen generations there were no few-shot prompts, so the value is null.
    Returns True if the file was patched.
    """
    path = Path(path)
    data = yaml.safe_load(path.read_text(encoding="utf-8"))

    if not isinstance(data, dict):
        raise ValueError(f"Expected YAML mapping in {path}")

    if "fsp_by_resp_id" in data:
        return False

    data["fsp_by_resp_id"] = None
    path.write_text(
        yaml.safe_dump(data, sort_keys=False, allow_unicode=True, width=120),
        encoding="utf-8",
    )
    return True


def read_json_or_jsonl(path: Path) -> list[dict[str, Any]]:
    """Read a JSON array or JSONL file into a list of dictionaries."""
    path = Path(path)
    text = path.read_text(encoding="utf-8").lstrip()
    if not text:
        return []
    if text[0] == "[":
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError(f"Expected JSON array in {path}")
        return data
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if not isinstance(obj, dict):
                raise ValueError(f"Expected JSON object on {path}:{line_no}")
            rows.append(obj)
    return rows


def write_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    """Write dictionaries to CSV, preserving all keys seen across rows."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    keys: list[str] = []
    seen = set()
    for row in rows:
        for key in row.keys():
            if key not in seen:
                seen.add(key)
                keys.append(key)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)


def read_json_or_jsonl(path: Path) -> list[dict[str, Any]]:
    """Read a JSON array or JSONL file into a list of dictionaries."""
    path = Path(path)
    text = path.read_text(encoding="utf-8").lstrip()
    if not text:
        return []
    if text[0] == "[":
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError(f"Expected JSON array in {path}")
        return data
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if not isinstance(obj, dict):
                raise ValueError(f"Expected JSON object on {path}:{line_no}")
            rows.append(obj)
    return rows


def write_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    """Write dictionaries to CSV, preserving all keys seen across rows."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    keys: list[str] = []
    seen = set()
    for row in rows:
        for key in row.keys():
            if key not in seen:
                seen.add(key)
                keys.append(key)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)


def expected_eval_path_for_response_yaml(response_yaml_path: Path) -> Path:
    """
    Compute the standard ChainScope eval YAML path for a converted response YAML.

    The converter writes:
      CONVERSION_OUT_DIR/responses/<sampling>/<pre_id>/<dataset_id>/<model>.yaml

    eval_cots.py normally writes:
      CHAIN_SCOPE_ROOT/chainscope/data/cot_eval/<instr_id>/<sampling>/<pre_id>/<dataset_id>/<model>.yaml
    """
    response_yaml_path = Path(response_yaml_path)
    rel = response_yaml_path.relative_to(CONVERSION_OUT_DIR / "responses")
    data = yaml.safe_load(response_yaml_path.read_text(encoding="utf-8"))
    instr_id = data.get("instr_id", INSTR_ID) if isinstance(data, dict) else INSTR_ID
    return CHAIN_SCOPE_ROOT / "chainscope" / "data" / "cot_eval" / instr_id / rel

def split_qid_direction(qid: str) -> tuple[str, str]:
    """Split '<pair_id>_fwd' or '<pair_id>_rev' into pair_id and direction."""
    qid = str(qid)
    if qid.endswith("_fwd"):
        return qid[:-4], "fwd"
    if qid.endswith("_rev"):
        return qid[:-4], "rev"
    raise ValueError(f"Could not infer direction from id/question_id: {qid!r}")


def detect_generation_schema(rows: list[dict[str, Any]]) -> str:
    """
    Identify which supported generation JSONL schema is being used.

    Supported schemas:
      - "pair_id_direction": rows contain pair_id, plus direction and/or question_id.
      - "cluster_id_sample": rows contain id='<pair_id>_fwd/_rev' and sample_id.
    """
    if not rows:
        raise ValueError("Input generations file is empty.")

    sample = next((r for r in rows if isinstance(r, dict)), None)
    if sample is None:
        raise ValueError("Input generations file contains no JSON objects.")

    if "pair_id" in sample:
        return "pair_id_direction"

    if "id" in sample and str(sample["id"]).endswith(("_fwd", "_rev")):
        return "cluster_id_sample"

    raise ValueError(
        "Unknown generation JSONL format. Expected either:\n"
        "  1. fields like pair_id, direction/question_id, raw_output; or\n"
        "  2. fields like id='<pair_id>_fwd/_rev', sample_id, raw_output."
    )


def normalize_generation_row_for_converter(row: dict[str, Any], schema: str) -> dict[str, Any]:
    """
    Normalize either supported JSONL format into the schema expected by
    convert_qwen_jsonl_to_chainscope_yaml.py.
    """
    out = dict(row)

    if schema == "pair_id_direction":
        pair_id = str(row.get("pair_id"))
        direction = row.get("direction")

        question_id = row.get("question_id") or row.get("id")
        if direction not in {"fwd", "rev"}:
            if question_id:
                _, direction = split_qid_direction(str(question_id))
            else:
                raise ValueError(
                    f"Row has pair_id={pair_id!r}, but no usable direction or question_id/id suffix."
                )

        if not question_id:
            question_id = f"{pair_id}_{direction}"

    elif schema == "cluster_id_sample":
        question_id = str(row.get("question_id") or row.get("id"))
        pair_id, direction = split_qid_direction(question_id)

    else:
        raise ValueError(f"Unsupported schema: {schema!r}")

    sample_value = (
        row.get("sample_idx")
        if row.get("sample_idx") is not None
        else row.get("sample_id")
        if row.get("sample_id") is not None
        else row.get("run_idx")
        if row.get("run_idx") is not None
        else row.get("seed")
    )

    out["pair_id"] = pair_id
    out["direction"] = direction
    out["question_id"] = str(question_id)

    # Keep response ids unique for files whose only sample index field is sample_id.
    if out.get("sample_idx") is None and sample_value is not None:
        out["sample_idx"] = sample_value
    if out.get("run_idx") is None and sample_value is not None:
        out["run_idx"] = sample_value
    if out.get("seed") is None and sample_value is not None:
        out["seed"] = sample_value

    # The converter has defaults for these, but adding them makes the normalized file explicit.
    out["model"] = str(row.get("model", "unknown-model"))
    out["temperature"] = float(row.get("temperature", 0.7))
    out["top_p"] = float(row.get("top_p", 0.9))
    out["max_new_tokens"] = int(row.get("max_new_tokens", row.get("max_tokens", 2000)))
    out["raw_output"] = str(row.get("raw_output", row.get("response", row.get("output", ""))))

    return out


def write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    """Write rows as UTF-8 JSONL."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def prepare_converter_input(responses_jsonl: Path, normalized_jsonl: Path) -> dict[str, Any]:
    """
    Read the input generations file, detect its schema, normalize rows, and write a
    converter-compatible JSONL.

    Returns metadata used by the conversion cell.
    """
    rows = read_json_or_jsonl(responses_jsonl)
    schema = detect_generation_schema(rows)

    normalized_rows = [
        normalize_generation_row_for_converter(row, schema)
        for row in rows
    ]
    write_jsonl(normalized_jsonl, normalized_rows)

    by_direction = Counter(row["direction"] for row in normalized_rows)
    by_model = Counter(row.get("model", "unknown-model") for row in normalized_rows)
    unique_pairs = {row["pair_id"] for row in normalized_rows}
    unique_questions = {row["question_id"] for row in normalized_rows}

    return {
        "schema": schema,
        "input_path": str(responses_jsonl),
        "normalized_path": str(normalized_jsonl),
        "rows_read": len(rows),
        "rows_written": len(normalized_rows),
        "unique_pairs": len(unique_pairs),
        "unique_questions": len(unique_questions),
        "direction_counts": dict(by_direction),
        "model_counts": dict(by_model),
    }


# Convert Qwen JSONL responses to ChainScope YAML, auto-detect input format, run `eval_cots.py`, and save analysis outputs

This notebook supports both JSONL formats:

1. `pair_id` / `direction` / `question_id` format.
2. cluster format with `id = "<pair_id>_fwd"` or `"<pair_id>_rev"` plus `sample_id`.

The conversion cell normalizes either format into a temporary converter-compatible JSONL, then runs the same converter and eval pipeline.

## 1. Configure conversion

In [ ]:
# Set this to the directory that contains:
#   - chainscope/
#   - convert_qwen_jsonl_to_chainscope_yaml.py
#   - selected_621_pairs.json
#   - dataX/
#
# If this notebook is already running from that directory, Path.cwd() is correct.
REPO_ROOT = Path.cwd().resolve()

# ChainScope checkout inside REPO_ROOT.
CHAIN_SCOPE_ROOT = REPO_ROOT / "chainscope"

# Make your local project and ChainScope importable.
add_to_sys_path(REPO_ROOT)
add_to_sys_path(CHAIN_SCOPE_ROOT)

# Converter script generated by ChatGPT. Put it in REPO_ROOT, or edit this path.
CONVERTER_PY = REPO_ROOT / "convert_qwen_jsonl_to_chainscope_yaml.py"

# Input files.
# Edit RESPONSES_JSONL to either supported JSONL format.
#RESPONSES_JSONL = REPO_ROOT / "data2" / "qwen2_5_7b_instruct_621pairs_10samples_cluster.jsonl"
RESPONSES_JSONL = REPO_ROOT / "data1" / "modal_qwen3_4b_backup_20260710_0015" / "outputs" / "qwen3_4b_instruct_621pairs_10samples_final.jsonl"
GOLD_JSON = REPO_ROOT / "selected_621_pairs.json"
EVAL_COTS_PY = REPO_ROOT / "chainscope" / "scripts" / "iphr" / "eval_cots.py"

# Conversion output.
# This can be anywhere writable. It does not have to be inside chainscope/data.
CONVERSION_OUT_DIR = REPO_ROOT / "data1_converted"

# Temporary converter-compatible JSONL written by the conversion cell.
NORMALIZED_RESPONSES_JSONL = CONVERSION_OUT_DIR / "_normalized_generations_for_converter.jsonl"

# Analysis output directory.
ANALYSIS_OUT_DIR = REPO_ROOT / "data1_analysis"

# Converter options.
INSTR_ID = "instr-wm"
CONVERSION_SUFFIX = "converted"
REPO_LAYOUT = False
WRITE_QUESTION_YAMLS = True

# Recommended for reruns after changing converter/schema.
# This deletes the conversion output directory before writing fresh YAMLs.
CLEAR_OUTPUT_DIR = True

print("REPO_ROOT:", REPO_ROOT)
print("CHAIN_SCOPE_ROOT:", CHAIN_SCOPE_ROOT)
print("CONVERTER_PY:", CONVERTER_PY)
print("RESPONSES_JSONL:", RESPONSES_JSONL)
print("GOLD_JSON:", GOLD_JSON)
print("EVAL_COTS_PY:", EVAL_COTS_PY)
print("CONVERSION_OUT_DIR:", CONVERSION_OUT_DIR)
print("NORMALIZED_RESPONSES_JSONL:", NORMALIZED_RESPONSES_JSONL)
print("ANALYSIS_OUT_DIR:", ANALYSIS_OUT_DIR)

## 2. Run conversion

In [ ]:
# Basic path checks before conversion.
required_paths = {
    "REPO_ROOT": REPO_ROOT,
    "CHAIN_SCOPE_ROOT": CHAIN_SCOPE_ROOT,
    "CONVERTER_PY": CONVERTER_PY,
    "RESPONSES_JSONL": RESPONSES_JSONL,
    "GOLD_JSON": GOLD_JSON,
}

missing = [f"{name}: {path}" for name, path in required_paths.items() if not Path(path).exists()]
if missing:
    raise FileNotFoundError("Missing required paths:\n" + "\n".join(missing))

if CLEAR_OUTPUT_DIR and CONVERSION_OUT_DIR.exists():
    shutil.rmtree(CONVERSION_OUT_DIR)
    print(f"Deleted old conversion output: {CONVERSION_OUT_DIR}")

CONVERSION_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Auto-detect and normalize either supported JSONL format.
normalization_summary = prepare_converter_input(
    responses_jsonl=RESPONSES_JSONL,
    normalized_jsonl=NORMALIZED_RESPONSES_JSONL,
)

print("Detected input schema:", normalization_summary["schema"])
print("Rows read:", normalization_summary["rows_read"])
print("Rows normalized:", normalization_summary["rows_written"])
print("Unique pairs:", normalization_summary["unique_pairs"])
print("Unique questions:", normalization_summary["unique_questions"])
print("Direction counts:", normalization_summary["direction_counts"])
print("Model counts:")
pprint(normalization_summary["model_counts"])
print("Normalized converter input:", NORMALIZED_RESPONSES_JSONL)

converter_input_path = Path(normalization_summary["normalized_path"])

converter_module = import_module_from_path("convert_qwen_jsonl_to_chainscope_yaml", CONVERTER_PY)

if hasattr(converter_module, "convert_qwen_jsonl_to_chainscope_yaml"):
    summary = converter_module.convert_qwen_jsonl_to_chainscope_yaml(
        responses_path=converter_input_path,
        gold_path=GOLD_JSON,
        out_dir=CONVERSION_OUT_DIR,
        instr_id=INSTR_ID,
        suffix=CONVERSION_SUFFIX,
        repo_layout=REPO_LAYOUT,
        write_question_yamls=WRITE_QUESTION_YAMLS,
    )
else:
    # Fallback for an older CLI-only converter.
    cmd = [
        sys.executable,
        str(CONVERTER_PY),
        "--responses", str(converter_input_path),
        "--gold", str(GOLD_JSON),
        "--out-dir", str(CONVERSION_OUT_DIR),
        "--instr-id", INSTR_ID,
        "--suffix", CONVERSION_SUFFIX,
    ]
    if REPO_LAYOUT:
        cmd.append("--repo-layout")
    if WRITE_QUESTION_YAMLS:
        cmd.append("--write-question-yamls")
    print("Converter module has no function; using CLI fallback:")
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    manifest_path = CONVERSION_OUT_DIR / "response_paths.txt"
    summary = {
        "rows_read": normalization_summary["rows_written"],
        "gold_pairs_loaded": None,
        "rows_skipped_unknown_pair_id": None,
        "cot_response_paths": [
            line.strip()
            for line in manifest_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ],
        "question_paths": [],
        "manifest_path": str(manifest_path),
    }

response_paths = [Path(p) for p in summary["cot_response_paths"]]
manifest_path = Path(summary["manifest_path"])

# Extra safety: patch already-written YAML files if an older converter omitted fsp_by_resp_id.
patched = [p for p in response_paths if ensure_fsp_by_resp_id(p)]

print(f"Read response rows: {summary['rows_read']}")
print(f"Loaded golden pairs: {summary['gold_pairs_loaded']}")
print(f"Skipped rows with unknown pair_id: {summary['rows_skipped_unknown_pair_id']}")
print(f"Wrote CotResponses YAML files: {len(response_paths)}")
print(f"Patched YAML files missing fsp_by_resp_id: {len(patched)}")
print(f"Wrote question YAML files: {len(summary.get('question_paths', []))}")
print(f"Manifest: {manifest_path}")

print("\nFirst few response paths:")
for p in response_paths[:5]:
    print(" ", p)

# Confirm every generated response YAML contains the required field.
missing_required = []
for p in response_paths:
    data = yaml.safe_load(Path(p).read_text(encoding="utf-8"))
    if not isinstance(data, dict) or "fsp_by_resp_id" not in data:
        missing_required.append(p)

if missing_required:
    raise RuntimeError("Some files still lack fsp_by_resp_id:\n" + "\n".join(map(str, missing_required)))

print("Confirmed fsp_by_resp_id exists in every generated CotResponses YAML.")


In [ ]:
# Validate generated YAMLs against your local ChainScope CotResponses dataclass.
# This catches schema mismatches before running eval_cots.py.
from chainscope.typing import CotResponses

loaded_responses = []
for p in response_paths:
    # Defensive patch in case this cell is rerun against older outputs.
    ensure_fsp_by_resp_id(p)
    loaded_responses.append(CotResponses.load(p))

print(f"Validated {len(loaded_responses)} CotResponses YAML files")

summary_rows = []
for cr, path in zip(loaded_responses, response_paths):
    n_questions = len(cr.responses_by_qid)
    n_responses = sum(len(v) for v in cr.responses_by_qid.values())
    summary_rows.append({
        "path": str(path),
        "model_id": cr.model_id,
        "prop_id": cr.ds_params.prop_id,
        "comparison": cr.ds_params.comparison,
        "answer": cr.ds_params.answer,
        "n_questions": n_questions,
        "n_responses": n_responses,
        "fsp_by_resp_id": getattr(cr, "fsp_by_resp_id", None),
    })

pprint(summary_rows[:10])


## 3. Configure eval

### API key file format

For OpenAI evaluation, create this file:

```text
openai_api_key.txt
```

in `REPO_ROOT`, with either of these formats:

```text
sk-...
```

or:

```text
OPENAI_API_KEY=sk-...
```

Keep this file out of git. Use `EVAL_BACKEND = "heuristic"` for local parsing without any API key, or `EVAL_BACKEND = "openai"` for the OpenAI API judge.


For Anthropic evaluation, create this file:

```text
anthropic_api_key.txt
```

in `REPO_ROOT`, with either:

```text
sk-ant-...
```

or:

```text
ANTHROPIC_API_KEY=sk-ant-...
```


In [ ]:
# Evaluation backend switch.
#
# Choose one:
#   "heuristic" -> local rule-based parsing; no API key needed
#   "openai"    -> OpenAI API judge; loads OPENAI_API_KEY from OPENAI_API_KEY_FILE
#   "anthropic" -> Anthropic API judge; loads ANTHROPIC_API_KEY from ANTHROPIC_API_KEY_FILE
EVAL_BACKEND = "openai"   # change to "openai" to use the OpenAI API

# OpenAI API-key file. The file can contain either:
#   sk-...
# or:
#   OPENAI_API_KEY=sk-...
OPENAI_API_KEY_FILE = REPO_ROOT / "openai_api_key.txt"
ANTHROPIC_API_KEY_FILE = REPO_ROOT / "anthropic_api_key.txt"

# ChainScope evaluator model alias for OpenAI.
# Use the alias supported by your local ChainScope repo.
# GPT4OL is the alias used previously for GPT-4o-latest-style evaluation.
OPENAI_EVALUATOR_MODEL_ID = "GPT4OL"
ANTHROPIC_EVALUATOR_MODEL_ID = "C3.7S"

# Eval options.
EVAL_TEST_MODE = False
EVAL_VERBOSE = True


def load_secret_from_file(path: Path) -> str:
    """
    Load a secret value from a local text file.

    Accepted file contents:
      sk-...
    or:
      OPENAI_API_KEY=sk-...

    Blank lines and lines starting with # are ignored.
    """
    path = Path(path).expanduser()

    if not path.exists():
        raise FileNotFoundError(f"Secret file not found: {path}")

    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()

        if not line or line.startswith("#"):
            continue

        if "=" in line:
            key, value = line.split("=", 1)
            if key.strip() == "OPENAI_API_KEY":
                value = value.strip().strip('"').strip("'")
                if value:
                    return value
        else:
            value = line.strip().strip('"').strip("'")
            if value:
                return value

    raise ValueError(f"No usable OPENAI_API_KEY found in {path}")


if EVAL_BACKEND == "heuristic":
    EVALUATOR_MODEL_ID = None
    EVAL_API = None
elif EVAL_BACKEND == "openai":
    EVALUATOR_MODEL_ID = OPENAI_EVALUATOR_MODEL_ID
    EVAL_API = "oai"

    # Load the API key from file and expose it only to this Python process.
    os.environ["OPENAI_API_KEY"] = load_secret_from_file(OPENAI_API_KEY_FILE)
elif EVAL_BACKEND == "anthropic":
    EVALUATOR_MODEL_ID = ANTHROPIC_EVALUATOR_MODEL_ID
    EVAL_API = "ant"

    # Load the API key from file and expose it only to this Python process.
    os.environ["ANTHROPIC_API_KEY"] = load_secret_from_file(ANTHROPIC_API_KEY_FILE)
else:
    raise ValueError(f"Unknown EVAL_BACKEND: {EVAL_BACKEND!r}. Use 'heuristic', 'openai', or 'anthropic'.")

# Use the response paths produced by the conversion step.
EVAL_RESPONSE_PATHS = response_paths

print(f"Configured {len(EVAL_RESPONSE_PATHS)} response YAML files for eval")
print("EVAL_BACKEND:", EVAL_BACKEND)
print("EVALUATOR_MODEL_ID:", EVALUATOR_MODEL_ID)
print("EVAL_API:", EVAL_API)
print("EVAL_TEST_MODE:", EVAL_TEST_MODE)
print("EVAL_VERBOSE:", EVAL_VERBOSE)
print("OPENAI_API_KEY present in process:", bool(os.environ.get("OPENAI_API_KEY")))

if EVAL_BACKEND == "openai":
    print("Loaded OpenAI key from:", OPENAI_API_KEY_FILE)
if EVAL_BACKEND == "anthropic":
    print("Loaded Anthropic key from:", ANTHROPIC_API_KEY_FILE)


## 4. Run `eval_cots.py`

In [ ]:
if not EVAL_COTS_PY.exists():
    raise FileNotFoundError(f"eval_cots.py not found: {EVAL_COTS_PY}")

# Defensive patch before eval in case files were edited between cells.
patched_before_eval = [p for p in EVAL_RESPONSE_PATHS if ensure_fsp_by_resp_id(p)]
if patched_before_eval:
    print(f"Patched {len(patched_before_eval)} files before eval.")

if EVAL_BACKEND == "openai" and not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError(
        "EVAL_BACKEND='openai' but OPENAI_API_KEY is not set. "
        "Check OPENAI_API_KEY_FILE in the eval config cell."
    )

eval_cots_module = import_module_from_path("chainscope_eval_cots_script", EVAL_COTS_PY)

# eval_cots.py uses click, so submit is usually a click Command.
# The actual Python function is available as submit.callback.
callback = getattr(eval_cots_module.submit, "callback", eval_cots_module.submit)
signature = inspect.signature(callback)

responses_arg = ",".join(str(p) for p in EVAL_RESPONSE_PATHS)

candidate_kwargs = {
    "responses_paths": responses_arg,
    "verbose": EVAL_VERBOSE,
    "llm_model_id": EVALUATOR_MODEL_ID,
    "api": EVAL_API,
    "test": EVAL_TEST_MODE,
}

call_kwargs = {
    name: value
    for name, value in candidate_kwargs.items()
    if name in signature.parameters
}

print("eval_cots callback signature:", signature)
print("Eval backend:", EVAL_BACKEND)
print("Calling eval_cots with kwargs:")
pprint(call_kwargs)

callback(**call_kwargs)

print("Evaluation finished.")


## 5. Configure analysis output

In [ ]:
# Analysis output configuration.
# The next cell writes all analysis outputs into this directory.
ANALYSIS_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional manual override. Leave empty for automatic discovery from response_paths.
# Example:
# MANUAL_EVAL_RESULT_PATHS = [Path("...")]
MANUAL_EVAL_RESULT_PATHS: list[Path] = []

# File outputs.
RESPONSE_LEVEL_CSV = ANALYSIS_OUT_DIR / "response_level_eval.csv"
QUESTION_LEVEL_CSV = ANALYSIS_OUT_DIR / "question_level_summary.csv"
PAIR_LEVEL_CSV = ANALYSIS_OUT_DIR / "pair_level_summary.csv"
SUMMARY_JSON = ANALYSIS_OUT_DIR / "analysis_summary.json"
REPORT_MD = ANALYSIS_OUT_DIR / "analysis_report.md"

print("Analysis output directory:", ANALYSIS_OUT_DIR)
print("Will write:")
for p in [RESPONSE_LEVEL_CSV, QUESTION_LEVEL_CSV, PAIR_LEVEL_CSV, SUMMARY_JSON, REPORT_MD]:
    print(" ", p)


## 6. Save analysis outputs

In [ ]:
# Load gold and raw response metadata.
gold_pairs = read_json_or_jsonl(GOLD_JSON)
gold_by_pair_id = {row["pair_id"]: row for row in gold_pairs}

raw_generation_rows = read_json_or_jsonl(RESPONSES_JSONL)

# Load the converted response YAMLs so we know which response IDs exist.
converted_response_rows = []
for response_yaml_path in response_paths:
    ensure_fsp_by_resp_id(response_yaml_path)
    data = yaml.safe_load(Path(response_yaml_path).read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        continue
    ds_params = data.get("ds_params", {}) or {}
    sampling_params = data.get("sampling_params", {}) or {}
    for qid, resp_by_id in (data.get("responses_by_qid", {}) or {}).items():
        pair_id = str(qid).removesuffix("_fwd").removesuffix("_rev")
        direction = "fwd" if str(qid).endswith("_fwd") else "rev" if str(qid).endswith("_rev") else None
        for response_id, response_text in (resp_by_id or {}).items():
            converted_response_rows.append({
                "response_yaml_path": str(response_yaml_path),
                "model_id": data.get("model_id"),
                "instr_id": data.get("instr_id"),
                "prop_id": ds_params.get("prop_id"),
                "comparison": ds_params.get("comparison"),
                "gold_answer": ds_params.get("answer"),
                "dataset_uuid": ds_params.get("uuid"),
                "dataset_suffix": ds_params.get("suffix"),
                "temperature": sampling_params.get("temperature"),
                "top_p": sampling_params.get("top_p"),
                "max_new_tokens": sampling_params.get("max_new_tokens"),
                "qid": qid,
                "pair_id": pair_id,
                "direction": direction,
                "response_id": response_id,
                "response_text": response_text,
            })

# Discover eval YAMLs efficiently.
# Do NOT recursively scan the whole ChainScope cot_eval tree; that repo directory can be huge.
if MANUAL_EVAL_RESULT_PATHS:
    eval_result_paths = [Path(p) for p in MANUAL_EVAL_RESULT_PATHS]
else:
    eval_result_paths = []
    for response_yaml_path in response_paths:
        expected_path = expected_eval_path_for_response_yaml(Path(response_yaml_path))
        if expected_path.exists():
            eval_result_paths.append(expected_path)

eval_result_paths = sorted(set(eval_result_paths))
print(f"Eval YAML files found for this run: {len(eval_result_paths)}")
if not eval_result_paths:
    print("No eval YAMLs found. Run the eval cell first, or set MANUAL_EVAL_RESULT_PATHS in the previous cell.")

# Parse eval results.
eval_rows = []
for eval_yaml_path in eval_result_paths:
    data = yaml.safe_load(Path(eval_yaml_path).read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        continue
    ds_params = data.get("ds_params", {}) or {}
    sampling_params = data.get("sampling_params", {}) or {}
    results_by_qid = data.get("results_by_qid", {}) or {}
    for qid, by_response_id in results_by_qid.items():
        pair_id = str(qid).removesuffix("_fwd").removesuffix("_rev")
        direction = "fwd" if str(qid).endswith("_fwd") else "rev" if str(qid).endswith("_rev") else None
        for response_id, result_obj in (by_response_id or {}).items():
            if isinstance(result_obj, dict):
                result = result_obj.get("result") or result_obj.get("final_answer")
                final_answer = result_obj.get("final_answer")
                equal_values = result_obj.get("equal_values")
                explanation_final_answer = result_obj.get("explanation_final_answer")
                explanation_equal_values = result_obj.get("explanation_equal_values")
            else:
                result = str(result_obj)
                final_answer = None
                equal_values = None
                explanation_final_answer = None
                explanation_equal_values = None

            eval_rows.append({
                "eval_yaml_path": str(eval_yaml_path),
                "evaluator": data.get("evaluator"),
                "model_id": data.get("model_id"),
                "instr_id": data.get("instr_id"),
                "prop_id": ds_params.get("prop_id"),
                "comparison": ds_params.get("comparison"),
                "gold_answer": ds_params.get("answer"),
                "dataset_uuid": ds_params.get("uuid"),
                "dataset_suffix": ds_params.get("suffix"),
                "temperature": sampling_params.get("temperature"),
                "top_p": sampling_params.get("top_p"),
                "max_new_tokens": sampling_params.get("max_new_tokens"),
                "qid": qid,
                "pair_id": pair_id,
                "direction": direction,
                "response_id": response_id,
                "result": result,
                "final_answer": final_answer,
                "equal_values": equal_values,
                "explanation_final_answer": explanation_final_answer,
                "explanation_equal_values": explanation_equal_values,
            })

# Save response-level rows even if eval did not run, so we keep useful converted-run metadata.
response_level_rows = eval_rows if eval_rows else converted_response_rows

# Question-level aggregation.
question_groups: dict[tuple[Any, ...], list[dict[str, Any]]] = defaultdict(list)
for row in eval_rows:
    key = (
        row.get("model_id"), row.get("prop_id"), row.get("comparison"), row.get("gold_answer"),
        row.get("qid"), row.get("pair_id"), row.get("direction"), row.get("temperature"), row.get("top_p"), row.get("max_new_tokens"),
    )
    question_groups[key].append(row)

question_summary_rows = []
for key, rows in sorted(question_groups.items(), key=lambda kv: str(kv[0])):
    model_id, prop_id, comparison, gold_answer, qid, pair_id, direction, temperature, top_p, max_new_tokens = key
    counts = Counter(str(r.get("result")) for r in rows)
    total = len(rows)
    yes = counts.get("YES", 0)
    no = counts.get("NO", 0)
    unknown = counts.get("UNKNOWN", 0)
    refused = counts.get("REFUSED", 0)
    correct = yes if gold_answer == "YES" else no if gold_answer == "NO" else 0
    majority_label = max(counts.items(), key=lambda kv: kv[1])[0] if counts else None
    question_summary_rows.append({
        "model_id": model_id,
        "prop_id": prop_id,
        "comparison": comparison,
        "gold_answer": gold_answer,
        "qid": qid,
        "pair_id": pair_id,
        "direction": direction,
        "temperature": temperature,
        "top_p": top_p,
        "max_new_tokens": max_new_tokens,
        "n": total,
        "yes_count": yes,
        "no_count": no,
        "unknown_count": unknown,
        "refused_count": refused,
        "p_yes": yes / total if total else None,
        "p_no": no / total if total else None,
        "p_unknown": unknown / total if total else None,
        "p_refused": refused / total if total else None,
        "p_correct": correct / total if total else None,
        "majority_label": majority_label,
    })

# Pair-level aggregation: join fwd/rev question summaries.
by_pair: dict[tuple[Any, ...], dict[str, dict[str, Any]]] = defaultdict(dict)
for row in question_summary_rows:
    key = (
        row.get("model_id"), row.get("prop_id"), row.get("comparison"),
        row.get("pair_id"), row.get("temperature"), row.get("top_p"), row.get("max_new_tokens"),
    )
    if row.get("direction") in {"fwd", "rev"}:
        by_pair[key][row["direction"]] = row

pair_summary_rows = []
for key, dirs in sorted(by_pair.items(), key=lambda kv: str(kv[0])):
    model_id, prop_id, comparison, pair_id, temperature, top_p, max_new_tokens = key
    fwd = dirs.get("fwd", {})
    rev = dirs.get("rev", {})
    gold_pair = gold_by_pair_id.get(pair_id, {})
    fwd_majority = fwd.get("majority_label")
    rev_majority = rev.get("majority_label")
    same_yes_no_majority = fwd_majority == rev_majority and fwd_majority in {"YES", "NO"}
    pair_summary_rows.append({
        "model_id": model_id,
        "prop_id": prop_id,
        "comparison": comparison,
        "pair_id": pair_id,
        "x_name": gold_pair.get("x_name"),
        "y_name": gold_pair.get("y_name"),
        "x_value": gold_pair.get("x_value"),
        "y_value": gold_pair.get("y_value"),
        "temperature": temperature,
        "top_p": top_p,
        "max_new_tokens": max_new_tokens,
        "fwd_gold": fwd.get("gold_answer"),
        "rev_gold": rev.get("gold_answer"),
        "fwd_n": fwd.get("n"),
        "rev_n": rev.get("n"),
        "fwd_p_yes": fwd.get("p_yes"),
        "rev_p_yes": rev.get("p_yes"),
        "fwd_p_no": fwd.get("p_no"),
        "rev_p_no": rev.get("p_no"),
        "fwd_p_correct": fwd.get("p_correct"),
        "rev_p_correct": rev.get("p_correct"),
        "fwd_majority_label": fwd_majority,
        "rev_majority_label": rev_majority,
        "same_yes_no_majority": same_yes_no_majority,
        "yes_yes_majority": same_yes_no_majority and fwd_majority == "YES",
        "no_no_majority": same_yes_no_majority and fwd_majority == "NO",
        "abs_p_yes_gap": abs((fwd.get("p_yes") or 0) - (rev.get("p_yes") or 0)) if fwd and rev else None,
        "abs_p_correct_gap": abs((fwd.get("p_correct") or 0) - (rev.get("p_correct") or 0)) if fwd and rev else None,
    })

# Capture evaluator backend metadata for saved files.
# These variables come from the eval configuration cell.
eval_backend_used = globals().get("EVAL_BACKEND")
eval_api_used = globals().get("EVAL_API")
evaluator_model_id_used = globals().get("EVALUATOR_MODEL_ID")
openai_evaluator_model_id_used = globals().get("OPENAI_EVALUATOR_MODEL_ID")
anthropic_evaluator_model_id_used = globals().get("ANTHROPIC_EVALUATOR_MODEL_ID")

if eval_backend_used is None:
    if eval_api_used == "oai":
        eval_backend_used = "openai"
    elif eval_api_used == "ant":
        eval_backend_used = "anthropic"
    elif evaluator_model_id_used is None and eval_api_used is None:
        eval_backend_used = "heuristic"
    else:
        eval_backend_used = str(eval_api_used or "unknown")

# Evaluator labels recorded in generated eval YAMLs, if any.
eval_yaml_evaluators = sorted({
    str(row.get("evaluator"))
    for row in eval_rows
    if row.get("evaluator") not in (None, "", "None")
})

# Write files.
write_csv(RESPONSE_LEVEL_CSV, response_level_rows)
write_csv(QUESTION_LEVEL_CSV, question_summary_rows)
write_csv(PAIR_LEVEL_CSV, pair_summary_rows)

summary_payload = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "responses_jsonl": str(RESPONSES_JSONL),
    "normalized_responses_jsonl": str(globals().get("NORMALIZED_RESPONSES_JSONL", "")),
    "input_schema": globals().get("normalization_summary", {}).get("schema"),
    "gold_json": str(GOLD_JSON),
    "conversion_out_dir": str(CONVERSION_OUT_DIR),
    "analysis_out_dir": str(ANALYSIS_OUT_DIR),
    "eval_backend": eval_backend_used,
    "eval_api": eval_api_used,
    "evaluator_model_id": evaluator_model_id_used,
    "openai_evaluator_model_id": openai_evaluator_model_id_used,
    "anthropic_evaluator_model_id": anthropic_evaluator_model_id_used,
    "eval_yaml_evaluators": eval_yaml_evaluators,
    "n_raw_generation_rows": len(raw_generation_rows),
    "n_gold_pairs": len(gold_pairs),
    "n_converted_response_rows": len(converted_response_rows),
    "n_eval_yaml_files": len(eval_result_paths),
    "n_eval_response_rows": len(eval_rows),
    "n_question_summaries": len(question_summary_rows),
    "n_pair_summaries": len(pair_summary_rows),
    "result_counts": dict(Counter(str(row.get("result")) for row in eval_rows)),
    "majority_same_answer_pairs": sum(1 for row in pair_summary_rows if row.get("same_yes_no_majority")),
    "yes_yes_majority_pairs": sum(1 for row in pair_summary_rows if row.get("yes_yes_majority")),
    "no_no_majority_pairs": sum(1 for row in pair_summary_rows if row.get("no_no_majority")),
    "files": {
        "response_level_csv": str(RESPONSE_LEVEL_CSV),
        "question_level_csv": str(QUESTION_LEVEL_CSV),
        "pair_level_csv": str(PAIR_LEVEL_CSV),
        "summary_json": str(SUMMARY_JSON),
        "report_md": str(REPORT_MD),
    },
}
SUMMARY_JSON.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False), encoding="utf-8")

report_lines = [
    "# Qwen IPHR analysis report",
    "",
    f"Created UTC: `{summary_payload['created_at_utc']}`",
    "",
    "## Inputs",
    f"- Responses JSONL: `{RESPONSES_JSONL}`",
    f"- Normalized converter JSONL: `{globals().get('NORMALIZED_RESPONSES_JSONL', '')}`",
    f"- Detected input schema: `{globals().get('normalization_summary', {}).get('schema', 'unknown')}`",
    f"- Gold JSON: `{GOLD_JSON}`",
    f"- Converted YAML directory: `{CONVERSION_OUT_DIR}`",
    "",
    "## Evaluator backend",
    f"- Backend switch: `{eval_backend_used}`",
    f"- ChainScope API backend: `{eval_api_used}`",
    f"- Evaluator model ID: `{evaluator_model_id_used}`",
    f"- OpenAI evaluator alias: `{openai_evaluator_model_id_used}`",
    f"- Anthropic evaluator alias: `{anthropic_evaluator_model_id_used}`",
    f"- Evaluator labels found in eval YAMLs: `{', '.join(eval_yaml_evaluators) if eval_yaml_evaluators else 'none / heuristic or not yet evaluated'}`",
    "",
    "## Counts",
    f"- Raw generation rows: {len(raw_generation_rows)}",
    f"- Gold pairs: {len(gold_pairs)}",
    f"- Converted response rows: {len(converted_response_rows)}",
    f"- Eval YAML files found: {len(eval_result_paths)}",
    f"- Eval response rows: {len(eval_rows)}",
    f"- Question summaries: {len(question_summary_rows)}",
    f"- Pair summaries: {len(pair_summary_rows)}",
    "",
    "## Result counts",
]
for label, count in sorted(summary_payload["result_counts"].items()):
    report_lines.append(f"- {label}: {count}")
report_lines += [
    "",
    "## Same-answer inverse-pair majority counts",
    f"- YES/YES or NO/NO majority pairs: {summary_payload['majority_same_answer_pairs']}",
    f"- YES/YES majority pairs: {summary_payload['yes_yes_majority_pairs']}",
    f"- NO/NO majority pairs: {summary_payload['no_no_majority_pairs']}",
    "",
    "## Written files",
    f"- Response-level CSV: `{RESPONSE_LEVEL_CSV}`",
    f"- Question-level CSV: `{QUESTION_LEVEL_CSV}`",
    f"- Pair-level CSV: `{PAIR_LEVEL_CSV}`",
    f"- Summary JSON: `{SUMMARY_JSON}`",
]
REPORT_MD.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

print("Saved analysis outputs:")
for p in [RESPONSE_LEVEL_CSV, QUESTION_LEVEL_CSV, PAIR_LEVEL_CSV, SUMMARY_JSON, REPORT_MD]:
    print(" ", p)

print("\nSummary:")
pprint(summary_payload)
